In [ ]:
%pip install statsmodels factor_analyzer

In [ ]:
# Identify highly correlated predictors and assess multicollinearity using Spearman correlation and VIF.
import sys, json; sys.path.append("..")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.treat import treat
from statsmodels.stats.outliers_influence import variance_inflation_factor
df = pd.read_parquet("../data/clean_v2.parquet")
fit = df[df.issue_d <= pd.Timestamp("2014-12-31")].copy()
spec = json.load(open("../reports/caps.json"))
X = treat(fit, spec); print(X.shape, "missing:", int(X.isna().sum().sum()))

corr = X.corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8)); im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90); ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
plt.colorbar(im); plt.tight_layout(); plt.savefig("../reports/spearman_matrix.png", dpi=110); plt.show()
pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
print((pairs.abs() > .7).sum(), "pairs with |rho| > 0.7"); print(pairs[pairs.abs() > .7].sort_values(key=abs, ascending=False).round(2).head(15))

Z = ((X - X.mean()) / X.std()).assign(const=1.0)
vif = pd.Series([variance_inflation_factor(Z.values, i) for i in range(X.shape[1])], index=X.columns).sort_values(ascending=False)
print(vif.round(1))

In [ ]:
# Apply PCA to assess dimensionality, explained variance, and the stability of treated versus raw features.
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from src.treat import HEAVY, RATIO
Xraw = X.copy()
for c in HEAVY + RATIO: Xraw[c] = fit[c].fillna(fit[c].median())
def run_pca(M):
    sc = StandardScaler().fit(M); return sc, PCA(random_state=0).fit(sc.transform(M))
sc_t, p_t = run_pca(X); sc_r, p_r = run_pca(Xraw)
ev = pd.DataFrame({"eigenvalue": p_t.explained_variance_, "var_treated": p_t.explained_variance_ratio_,
                   "cum_treated": p_t.explained_variance_ratio_.cumsum(), "var_raw": p_r.explained_variance_ratio_})
print(ev.head(10).round(3)); print("eigenvalue > 1:", int((p_t.explained_variance_ > 1).sum()))
M = np.abs(p_t.components_[:6] @ p_r.components_.T)
print("best-match |cosine|, treated PC1..PC6 vs raw:", M.max(axis=1).round(3), "| matched raw PCs:", M.argmax(axis=1) + 1)
print(pd.DataFrame(p_t.components_[:4].T, index=X.columns, columns=["PC1", "PC2", "PC3", "PC4"]).round(2))
plt.plot(range(1, 22), p_t.explained_variance_, "o-"); plt.axhline(1, ls="--"); plt.xlabel("component"); plt.ylabel("eigenvalue")
plt.title("Scree plot (treated)"); plt.savefig("../reports/scree.png", dpi=110); plt.show()

In [ ]:
# Validate factor-analysis suitability and determine the number of components using KMO, Bartlett's test, and parallel analysis.
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
chi2, pval = calculate_bartlett_sphericity(X); kmo_vars, kmo_all = calculate_kmo(X)
print(f"Bartlett chi2 = {chi2:.0f}, p = {pval:.3g} | overall KMO = {kmo_all:.3f}")
print(pd.Series(kmo_vars, index=X.columns).round(2).sort_values().head(6))
rng = np.random.default_rng(0)
sim = np.array([np.sort(np.linalg.eigvalsh(np.corrcoef(rng.standard_normal(X.shape), rowvar=False)))[::-1] for _ in range(100)])
thr = np.percentile(sim, 95, axis=0)
print(pd.DataFrame({"observed": p_t.explained_variance_[:10], "random_95%": thr[:10]}).round(3))
print("components above random:", int((p_t.explained_variance_ > thr).sum()))
gap = X.tot_cur_bal - X.avg_cur_bal
print("log(total/avg) vs log(accounts):", round(gap.corr(np.log(fit.open_acc.clip(lower=1)), method="spearman"), 2), round(gap.corr(np.log(fit.total_acc.clip(lower=1)), method="spearman"), 2))

In [ ]:
import factor_analyzer.factor_analyzer as fam
from sklearn.utils import check_array as _check_array
def _compat(X, *args, force_all_finite=True, **kwargs):
    return _check_array(X, *args, ensure_all_finite=force_all_finite, **kwargs)
fam.check_array = _compat

In [ ]:
# Refine factor analysis, determine six underlying factors, and evaluate loadings, communalities, and explained variance.
from factor_analyzer import FactorAnalyzer
X2 = X.drop(columns=["delinq_2yrs", "avg_cur_bal"])
chi2, p = calculate_bartlett_sphericity(X2); kv, ka = calculate_kmo(X2)
print(f"{X2.shape[1]} fields | Bartlett p = {p:.3g} | overall KMO = {ka:.3f} | lowest variable KMO = {kv.min():.2f}")
ev2 = np.sort(np.linalg.eigvalsh(np.corrcoef(X2, rowvar=False)))[::-1]
sim2 = np.array([np.sort(np.linalg.eigvalsh(np.corrcoef(rng.standard_normal(X2.shape), rowvar=False)))[::-1] for _ in range(100)])
k = int((ev2 > np.percentile(sim2, 95, axis=0)).sum()); print("factors from parallel analysis:", k)
fa = FactorAnalyzer(n_factors=k, rotation="varimax", method="minres").fit(X2)
fo = FactorAnalyzer(n_factors=k, rotation="oblimin", method="minres").fit(X2)
print("largest correlation between factors under oblimin:", round(float(np.abs(fo.phi_ - np.eye(k)).max()), 2))
L = pd.DataFrame(fa.loadings_, index=X2.columns, columns=[f"F{i+1}" for i in range(k)]); L["communality"] = fa.get_communalities()
print(L.round(2).to_string())
print("variance by factor:", fa.get_factor_variance()[1].round(3), "| total:", round(float(fa.get_factor_variance()[2][-1]), 3))

In [ ]:
# Compare Varimax and Oblimin factor structures and interpret correlated financial-risk dimensions.
Lo = pd.DataFrame(fo.loadings_, index=X2.columns, columns=L.columns[:-1])
print(pd.DataFrame(fo.phi_, index=Lo.columns, columns=Lo.columns).round(2))
a, b = L.drop(columns="communality").values, Lo.values
cong = np.abs(a.T @ b) / np.sqrt((a**2).sum(0)[:, None] * (b**2).sum(0)[None, :])
print("best-match congruence varimax vs oblimin:", cong.max(axis=1).round(3), "| matched oblimin factor:", cong.argmax(axis=1) + 1)
print(Lo.round(2).to_string())

In [ ]:
# Evaluate factor-model fit and validate the relationships between extracted factors.
obs = np.corrcoef(X2, rowvar=False); off = ~np.eye(len(obs), dtype=bool)
Pv, Po = L.drop(columns="communality").values, Lo.values
for name, R in [("varimax", Pv @ Pv.T), ("oblimin", Po @ fo.phi_ @ Po.T)]:
    r = np.abs(obs - R)[off]; print(f"{name}: max |residual| = {r.max():.3f} | mean = {r.mean():.3f}")
i, j = list(X2.columns).index("pub_rec"), list(X2.columns).index("tot_cur_bal")
print("pub_rec ~ tot_cur_bal | observed:", round(obs[i, j], 2), "| varimax implies:", round((Pv @ Pv.T)[i, j], 2), "| oblimin implies:", round((Po @ fo.phi_ @ Po.T)[i, j], 2))
print(pd.DataFrame(fo.transform(X2), columns=Lo.columns).corr().round(2))

In [ ]:
# compute the risk factor scores and store them for the contract
names = ["credit_footprint", "revolving_capacity", "utilization_stress", "recent_activity", "public_records", "debt_burden"]
R = np.corrcoef(X2, rowvar=False); W = np.linalg.solve(R, Pv)
Z = (X2 - fa.mean_) / fa.std_
S = pd.DataFrame(Z.values @ W, columns=names, index=X2.index)
try: print("max |manual - library|:", float(np.abs(S.values - fa.transform(X2)).max()))
except Exception as e: print("library comparison skipped:", type(e).__name__, e)
print(S.corr().round(2)); print(S.describe().loc[["mean", "std"]].round(2))
vi = fa.get_factor_variance()[1]
factors = [{"name": n, "method": "factor_analysis_varimax_regression_scores", "variance_explained": round(float(vi[i]), 3),
            "top_loadings": {c: round(float(L.loc[c, f"F{i+1}"]), 2) for c in L[f"F{i+1}"].abs().nlargest(3).index}} for i, n in enumerate(names)]
json.dump({"fields": list(X2.columns), "mean": [float(v) for v in fa.mean_], "std": [float(v) for v in fa.std_],
           "weights": W.tolist(), "factors": factors}, open("../reports/factor_spec.json", "w"), indent=1)
fs = json.load(open("../reports/factor_spec.json"))
S2 = ((X2 - np.array(fs["mean"])) / np.array(fs["std"])).values @ np.array(fs["weights"])
print("parity from stored file:", float(np.abs(S2 - S.values).max()))

In [ ]:
# Assess population stability between 2013 and 2014 using PSI and compare default rates.
f13, f14 = fit[fit.issue_d.dt.year == 2013], fit[fit.issue_d.dt.year == 2014]
def psi(a, b, bins=10):
    e = np.unique(np.quantile(a.dropna(), np.linspace(0, 1, bins + 1))); e[0], e[-1] = -np.inf, np.inf
    pa = np.histogram(a.dropna(), e)[0] / a.notna().sum(); pb = np.histogram(b.dropna(), e)[0] / b.notna().sum()
    pa, pb = np.clip(pa, 1e-4, None), np.clip(pb, 1e-4, None); return float(((pb - pa) * np.log(pb / pa)).sum())
T13, T14 = treat(f13, spec), treat(f14, spec)
F13 = pd.DataFrame(((T13[X2.columns] - np.array(fs["mean"])) / np.array(fs["std"])).values @ np.array(fs["weights"]), columns=names)
F14 = pd.DataFrame(((T14[X2.columns] - np.array(fs["mean"])) / np.array(fs["std"])).values @ np.array(fs["weights"]), columns=names)
res = {c: psi(T13[c], T14[c]) for c in T13.columns}; res.update({n: psi(F13[n], F14[n]) for n in names})
res.update({c: psi(f13[c], f14[c]) for c in ["int_rate", "term", "loan_amnt", "installment"]})
print(pd.Series(res).sort_values(ascending=False).round(3).head(15).to_string())
print("default rate 2013 vs 2014:", round(f13.defaulted.mean(), 3), round(f14.defaulted.mean(), 3))

In [ ]:
# Verify factor-score dimensions and alignment with the original training records.
print(S.shape, fit.shape, S.index.equals(fit.index))

In [ ]:
# Evaluate factor-level default risk across deciles using confidence intervals and statistical tests.
from scipy.stats import mannwhitneyu, spearmanr
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h
out = []
for n in S.columns:
    dec = pd.qcut(S[n].rank(method="first"), 10, labels=False)
    g = fit.groupby(dec).defaulted.agg(["sum", "count"]); r = g["sum"] / g["count"]
    lo, hi = wilson(g["sum"].values, g["count"].values)
    p = mannwhitneyu(S[n][fit.defaulted == 1], S[n][fit.defaulted == 0]).pvalue
    out.append({"factor": n, "decile 1 (lowest)": f"{r.iloc[0]:.3f} [{lo[0]:.3f}, {hi[0]:.3f}]",
                "decile 10 (highest)": f"{r.iloc[9]:.3f} [{lo[9]:.3f}, {hi[9]:.3f}]",
                "trend_rho": round(spearmanr(range(10), r)[0], 2), "mann_whitney_p": f"{p:.2g}"})
pd.DataFrame(out).set_index("factor")

In [ ]:
# Compare factor scores with credit grades and evaluate their incremental predictive value for default risk.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
y = fit.defaulted.values; G = pd.get_dummies(fit.grade, dtype=float).values
gnum = fit.grade.map({g: i for i, g in enumerate("ABCDEFG")}).values
print(pd.Series({n: round(np.corrcoef(S[n].values, gnum)[0, 1], 2) for n in S.columns}, name="corr with grade (A=0..G=6)"))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
def cvauc(M):
    p = cross_val_predict(LogisticRegression(max_iter=2000), M, y, cv=cv, method="predict_proba")[:, 1]
    return round(roc_auc_score(y, p), 4)
print("CV AUC | grade only:", cvauc(G), "| factors only:", cvauc(S.values), "| grade + factors:", cvauc(np.hstack([G, S.values])))
band = fit.grade.map(lambda g: "A-B" if g in "AB" else "C" if g == "C" else "D-G")
for n in ["recent_activity", "utilization_stress"]:
    t = pd.qcut(S[n].rank(method="first"), 3, labels=["low", "mid", "high"])
    print(n); print(fit.groupby([band, t], observed=True).defaulted.mean().unstack().round(3))
print(fit.groupby(fit.pub_rec > 0).defaulted.agg(["mean", "count"]).round(3))
w = pd.Series(np.array(fs["weights"])[:, 4], index=fs["fields"]); print(w[w.abs().nlargest(5).index].round(2))

In [ ]:
# Analyze geographic differences in default rates and assess the reliability of state- and ZIP-level estimates.
from scipy.stats import chi2_contingency
st = fit.groupby("addr_state").defaulted.agg(["sum", "count"]); st["rate"] = st["sum"] / st["count"]
st["lo"], st["hi"] = wilson(st["sum"].values, st["count"].values)
big = st[st["count"] >= 300].sort_values("rate")
print(big.round(3).iloc[[0, 1, 2, -3, -2, -1]][["count", "rate", "lo", "hi"]])
print("chi-square across states (>=300 loans): p =", round(chi2_contingency(pd.crosstab(fit.addr_state[fit.addr_state.isin(big.index)], fit.defaulted))[1], 4))
z = fit.zip_code.value_counts()
print("zip3 areas:", len(z), "| median loans per zip3:", int(z.median()), "| share with < 30 loans:", round((z < 30).mean(), 2))